In [ ]:
# Loading Dataset from Huggingface & Imports
# 0. Imports
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Loading Dataset
df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv")
print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [ ]:
# Dataset Description (Hady)
# 2. Size and Shape
print(f"Dataset Shape: {df.shape}")
print(f"Number of Records: {len(df)}")
print(f"Number of Features: {len(df.columns)}")

# 3. Feature Types and Names
print("\nFeature Information:")
print(df.info())

# 4. Class Labels and Distribution
print("\nClass Distribution:")
print(df['track_genre'].value_counts().sort_values(ascending=True))

# 5. Descriptive Statistics
print("\nDescriptive Statistics for Features:")
stats = df[['popularity', 'duration_ms', 'danceability', 'energy',
            'loudness', 'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo']].describe()
print(stats)
# ---

Dataset Shape: (114000, 21)
Number of Records: 114000
Number of Features: 21

Feature Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness     

In [ ]:
# Preprocessing (Hady)
# 6. Check for Missing Values
print("\nMissing Values per Feature:")
print(df.isnull().sum())

# Finding and printing the rows with missing values
missing_row = df[df.isnull().any(axis=1)]
print("\nRecord/s with missing values:")
print(missing_row)

# Dropping missing record
df = df.dropna()

# Verifying that the record is gone
print(f"\nDataset shape: {df.shape}")
print(f"Total missing values remaining: {df.isnull().sum().sum()}")
# ---


Missing Values per Feature:
Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

Record/s with missing values:
       Unnamed: 0                track_id artists album_name track_name  \
65900       65900  1kR4gIb7nGxHPI3D2ifs59     NaN        NaN        NaN   

       popularity  duration_ms  explicit  danceability  energy  ...  loudness  \
65900           0            0     False         0.501   0.583  ...     -9.46   

       mode  speechiness  acousticness  instrumentalness  liveness  valence  \
65900     0       0.0605          0.69           0.00396    0.0747    0.

In [ ]:
# 7. Dropping metadata and csv id/index columns
# List of columns to be removed
cols_to_drop = ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name']

# Dropping the metadata columns
df = df.drop(columns=cols_to_drop)

# Verifying the remaining features
print("Remaining columns in the dataset:")
print(df.columns.tolist())
print(f"\nShape after dropping columns: {df.shape}")
# ---

Remaining columns in the dataset:
['popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Shape after dropping columns: (113999, 16)


In [ ]:
# 8. Binary Encoding
# Convert 'explicit' to binary (0 and 1)
df['explicit'] = df['explicit'].astype(int)
# df['mode'] = df['mode'].astype(int) - does not require conversion as is already in binary format

# Verifying the conversion
print("Values after conversion:")
print(df['explicit'].value_counts())
print(df['mode'].value_counts())
# ---

Values after conversion:
explicit
0    104252
1      9747
Name: count, dtype: int64
mode
1    72681
0    41318
Name: count, dtype: int64


In [ ]:
# 9. One-hot encoding
# Columns to encode
X = df[['key', 'time_signature']]

# Creating the encoder, fitting and transforming data
enc = preprocessing.OneHotEncoder()
encoded_array = enc.fit_transform(X).toarray()

# To extract the feature names
new_columns = enc.get_feature_names_out(['key', 'time_signature'])

# Creating a new DataFrame with the array
encoded_df = pd.DataFrame(encoded_array, columns=new_columns, index=df.index)

# Dropping the original columns and attaching the new encoded ones
df = df.drop(columns=['key', 'time_signature'])
df = pd.concat([df, encoded_df], axis=1)

print(f"Dataset shape: {df.shape}")
# ---

Dataset shape: (113999, 31)


In [ ]:
# 10. Label Encoding
# Creating the LabelEncoder object
le = preprocessing.LabelEncoder()

# Fitting the encoder and transforming the track_genre column
df['track_genre'] = le.fit_transform(df['track_genre'])

# Verifying the transformation
print(f"\nTotal unique classes: {df['track_genre'].nunique()}")
print(f"Minimum label ID: {df['track_genre'].min()}")
print(f"Maximum label ID: {df['track_genre'].max()}")
# ---


Total unique classes: 114
Minimum label ID: 0
Maximum label ID: 113


In [ ]:
# 11. Scaling
# Listing all continuous features
continuous_features = [
    'popularity', 'duration_ms', 'danceability', 'energy',
    'loudness', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

# Initializing and apply the scaler
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])
# ---

In [ ]:
# 12. Train-Test Split
from sklearn.model_selection import train_test_split

# Separate features and target label
X = df.drop('track_genre', axis=1)
y = df['track_genre']

# Split into training and testing sets (80/20 split) with stratification on the label to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# 13. Baseline Model Training and Evaluation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

models = {
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gaussian Naive Bayes': GaussianNB()
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    results[name] = {'accuracy': acc, 'macro_f1': macro_f1}
    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} Macro F1-score: {macro_f1:.4f}")
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"{name} Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

print("\nBaseline model results:")
for model_name, metrics in results.items():
    print(f"{model_name}: Accuracy = {metrics['accuracy']:.4f}, Macro F1 = {metrics['macro_f1']:.4f}")


Training set shape: (91199, 30)
Test set shape: (22800, 30)

Training KNN...
KNN Accuracy: 0.2009
KNN Macro F1-score: 0.1985

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.07      0.23      0.11       200
           1       0.11      0.27      0.15       200
           2       0.03      0.10      0.05       200
           3       0.08      0.13      0.10       200
           4       0.17      0.31      0.22       200
           5       0.07      0.17      0.10       200
           6       0.24      0.47      0.32       200
           7       0.18      0.43      0.26       200
           8       0.08      0.17      0.11       200
           9       0.03      0.07      0.04       200
          10       0.17      0.27      0.21       200
          11       0.07      0.11      0.08       200
          12       0.08      0.18      0.11       200
          13       0.30      0.46      0.36       200
          14       0.27      0.34   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 14. Hyperparameter Tuning Examples
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
rf = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='f1_macro', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

print('Best Random Forest parameters:', grid_search_rf.best_params_)
print('Best cross-validated Macro F1 for Random Forest:', grid_search_rf.best_score_)

# Evaluate the tuned model on the test set
best_rf = grid_search_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
print('Random Forest Test Accuracy:', accuracy_score(y_test, y_pred_best_rf))
print('Random Forest Test Macro F1:', f1_score(y_test, y_pred_best_rf, average='macro'))

# Hyperparameter tuning for K-Nearest Neighbors
param_grid_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # Minkowski metric (1=Manhattan, 2=Euclidean)
}
knn = KNeighborsClassifier()
grid_search_knn = GridSearchCV(knn, param_grid_knn, cv=3, scoring='f1_macro', n_jobs=-1)
grid_search_knn.fit(X_train, y_train)

print('Best KNN parameters:', grid_search_knn.best_params_)
print('Best cross-validated Macro F1 for KNN:', grid_search_knn.best_score_)

best_knn = grid_search_knn.best_estimator_
y_pred_best_knn = best_knn.predict(X_test)
print('KNN Test Accuracy:', accuracy_score(y_test, y_pred_best_knn))
print('KNN Test Macro F1:', f1_score(y_test, y_pred_best_knn, average='macro'))
